In [2]:
"""
Модуль для создания CSV саммари из английских вакансий
На входе: текст вакансий на английском
На выходе: CSV с аналитическим саммари
"""

import pandas as pd
import time
import re
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, asdict
import hashlib
from datetime import datetime
import os


@dataclass
class VacancySummary:
    """Структура для хранения саммари вакансии"""
    vacancy_id: str
    vacancy_title: str
    extracted_requirements: str
    extracted_technologies: str
    extracted_experience: str
    key_skills: str
    summary_english: str
    summary_russian: str
    salary_info: str
    location: str
    work_type: str
    confidence_score: float
    processing_date: str
    source_text_hash: str
    source_text_length: int
    processing_time: float

    def to_dict(self) -> Dict:
        """Преобразует в словарь для DataFrame"""
        return asdict(self)


class VacancyToCSVSummarizer:
    """
    Создает CSV саммари из текстов вакансий на английском
    """

    def __init__(self, verbose: bool = True):
        """
        Инициализирует суммаризатор

        Args:
            verbose: Выводить ли прогресс обработки
        """
        self.verbose = verbose
        self._setup_keywords()

        if self.verbose:
            print("[ИНИЦИАЛИЗАЦИЯ] Суммаризатор вакансий готов")
            print("[ЯЗЫК] Обработка английских вакансий")

    def _setup_keywords(self):
        """Настраивает ключевые слова для анализа"""
        # Ключевые слова для поиска важной информации
        self.requirement_keywords = [
            "required", "requirements", "must have", "should have", "qualifications",
            "mandatory", "essential", "prerequisite", "looking for", "seeking"
        ]

        self.technology_keywords = [
            "python", "java", "javascript", "typescript", "react", "angular", "vue",
            "docker", "kubernetes", "aws", "azure", "gcp", "postgresql", "mysql",
            "mongodb", "redis", "graphql", "rest api", "node.js", "django", "flask",
            "spring", ".net", "c#", "c++", "go", "rust", "php", "ruby", "swift",
            "kotlin", "tensorflow", "pytorch", "pandas", "numpy", "scikit-learn",
            "spark", "hadoop", "kafka", "rabbitmq", "elasticsearch", "terraform",
            "ansible", "jenkins", "gitlab", "github actions"
        ]

        self.experience_keywords = [
            "experience", "years", "year", "yr", "senior", "junior", "mid-level",
            "entry-level", "expert", "advanced", "intermediate", "minimum",
            "at least", "proven", "demonstrated", "hands-on"
        ]

        self.salary_keywords = [
            "salary", "compensation", "pay", "rate", "budget", "range",
            "$", "€", "£", "USD", "EUR", "GBP", "per year", "annually",
            "hourly", "monthly", "competitive salary"
        ]

        self.location_keywords = [
            "remote", "onsite", "hybrid", "flexible", "office",
            "location", "based in", "work from", "relocation"
        ]

        # Маркетинговые фразы для фильтрации
        self.marketing_phrases = [
            "we offer", "we provide", "benefits", "perks", "insurance",
            "competitive salary", "bonus", "stock options", "relocation",
            "learning", "development", "growth opportunities", "work-life balance",
            "what we offer", "our benefits", "package includes", "compensation package",
            "team culture", "great atmosphere", "dynamic environment"
        ]

        # Распознавание типов работы
        self.work_type_patterns = {
            "full-time": ["full.time", "full time", "ft", "full-time"],
            "part-time": ["part.time", "part time", "pt", "part-time"],
            "contract": ["contract", "freelance", "temporary", "temp"],
            "internship": ["internship", "intern", "trainee", "graduate"]
        }

    def create_csv_summary(self, vacancies: List[Dict[str, Any]],
                          output_file: str = "vacancy_summaries.csv") -> pd.DataFrame:
        """
        Основная функция: создает CSV саммари из списка вакансий

        Args:
            vacancies: Список словарей с вакансиями
                Формат: [{"id": "vac_1", "title": "...", "text": "..."}, ...]
            output_file: Имя файла для сохранения CSV

        Returns:
            DataFrame с результатами
        """
        if self.verbose:
            print(f"СТАРТ ОБРАБОТКИ: {len(vacancies)} вакансий")

        csv_rows = []
        processed_count = 0
        error_count = 0

        for i, vacancy in enumerate(vacancies, 1):
            vacancy_id = vacancy.get('id', f'vacancy_{i:03d}')
            title = vacancy.get('title', 'No Title')
            text = vacancy.get('text', vacancy.get('full_text', ''))

            if self.verbose:
                print(f"\n[{i}/{len(vacancies)}] Обрабатываю: {title}")
                print(f"   ID: {vacancy_id}, Длина текста: {len(text)} символов")

            if not text or len(text) < 50:
                if self.verbose:
                    print("     Текст слишком короткий, пропускаю")
                error_count += 1
                continue

            start_time = time.time()

            try:
                # Извлекаем данные из вакансии
                extracted_data = self._analyze_vacancy(text, vacancy_id, title)

                # Создаем CSV строку
                csv_row = self._create_csv_row(extracted_data, text, start_time)
                csv_rows.append(csv_row)

                processed_count += 1

                if self.verbose:
                    confidence = extracted_data['confidence']
                    print(f"    Успешно. Уверенность: {confidence:.2%}")

            except Exception as e:
                if self.verbose:
                    print(f"    Ошибка: {str(e)[:100]}")
                error_count += 1

                # Создаем строку с ошибкой
                error_row = self._create_error_row(vacancy_id, title, text, str(e), start_time)
                csv_rows.append(error_row)

            # Небольшая пауза между обработкой
            if i < len(vacancies):
                time.sleep(0.1)

        # Создаем DataFrame
        if csv_rows:
            df = pd.DataFrame([row.to_dict() for row in csv_rows])
        else:
            df = pd.DataFrame()

        # Сохраняем в CSV
        if len(df) > 0:
            self._save_to_csv(df, output_file)

        # Выводим статистику
        self._print_statistics(processed_count, error_count, df)

        return df

    def _analyze_vacancy(self, text: str, vacancy_id: str, title: str) -> Dict[str, Any]:
        """Анализирует текст вакансии и извлекает структурированные данные"""
        text_lower = text.lower()

        # 1. Извлечение требований
        requirements = self._extract_requirements(text)

        # 2. Извлечение технологий
        technologies = self._extract_technologies(text_lower)

        # 3. Извлечение опыта
        experience = self._extract_experience(text_lower)

        # 4. Извлечение информации о зарплате
        salary_info = self._extract_salary_info(text)

        # 5. Извлечение локации и типа работы
        location, work_type = self._extract_location_worktype(text_lower)

        # 6. Извлечение ключевых навыков
        key_skills = self._extract_key_skills(text_lower)

        # 7. Генерация саммари на английском
        english_summary = self._generate_english_summary(
            title, requirements, technologies, experience, salary_info
        )

        # 8. Генерация саммари на русском
        russian_summary = self._generate_russian_summary(english_summary)

        # 9. Расчет уверенности
        confidence = self._calculate_confidence(
            requirements, technologies, experience, key_skills
        )

        return {
            'vacancy_id': vacancy_id,
            'title': title,
            'requirements': requirements,
            'technologies': technologies,
            'experience': experience,
            'key_skills': key_skills,
            'salary_info': salary_info,
            'location': location,
            'work_type': work_type,
            'english_summary': english_summary,
            'russian_summary': russian_summary,
            'confidence': confidence
        }

    def _extract_requirements(self, text: str) -> List[str]:
        """Извлекает требования из текста вакансии"""
        requirements = []
        lines = text.split('\n')

        # Ищем разделы с требованиями
        requirements_section = False

        for line in lines:
            line_lower = line.lower()

            # Проверяем начало раздела требований
            if any(keyword in line_lower for keyword in self.requirement_keywords):
                requirements_section = True
                continue

            # Если мы в разделе требований и это пункт списка
            if requirements_section:
                if line.strip().startswith(('-', '•', '*', '>', '–')):
                    requirement = line.strip()[1:].strip()
                    if requirement and len(requirement) > 5:
                        # Фильтруем маркетинговые фразы
                        if not any(phrase in requirement.lower() for phrase in self.marketing_phrases):
                            requirements.append(requirement)
                elif line.strip() and len(line.strip()) < 150:
                    # Непунктирные требования
                    if not any(phrase in line_lower for phrase in self.marketing_phrases):
                        requirements.append(line.strip())

            # Сброс при пустой строке
            if line.strip() == '':
                requirements_section = False

        # Если не нашли в структурированном виде, ищем по ключевым словам
        if not requirements:
            sentences = re.split(r'[.!?]+', text)
            for sentence in sentences:
                sentence = sentence.strip()
                if len(sentence) < 10:
                    continue

                sentence_lower = sentence.lower()
                if any(keyword in sentence_lower for keyword in self.requirement_keywords):
                    if not any(phrase in sentence_lower for phrase in self.marketing_phrases):
                        requirements.append(sentence)

        return requirements[:10]  # Ограничиваем количество

    def _extract_technologies(self, text_lower: str) -> List[str]:
        """Извлекает технологии из текста"""
        technologies = []

        for tech in self.technology_keywords:
            if tech in text_lower:
                # Правильное написание
                if tech == "c++":
                    technologies.append("C++")
                elif tech == "c#":
                    technologies.append("C#")
                elif tech == "node.js":
                    technologies.append("Node.js")
                elif tech == "rest api":
                    technologies.append("REST API")
                elif tech == "aws":
                    technologies.append("AWS")
                elif tech == "azure":
                    technologies.append("Azure")
                elif tech == "gcp":
                    technologies.append("GCP")
                else:
                    technologies.append(tech.capitalize())

        return list(set(technologies))  # Удаляем дубликаты

    def _extract_experience(self, text_lower: str) -> str:
        """Извлекает информацию об опыте работы"""
        experience_info = []

        # Поиск упоминаний опыта в годах
        years_patterns = [
            r'(\d+[\+\s]?(?:years?|yrs?|y\.?))',
            r'(\d+)\+?\s*years?\s*experience',
            r'(\d+)\+?\s*years?\s*of\s*experience',
            r'(\d+)\+?\s*years?\s*in\s*[a-zA-Z\s]+',
            r'minimum\s+of\s+(\d+)\s+years?',
            r'at\s+least\s+(\d+)\s+years?'
        ]

        for pattern in years_patterns:
            matches = re.findall(pattern, text_lower)
            if matches:
                for match in matches:
                    if isinstance(match, tuple):
                        match = match[0]
                    if match not in experience_info:
                        experience_info.append(match)

        # Поиск уровня позиции
        levels = ["senior", "junior", "mid-level", "entry-level", "lead", "principal"]
        for level in levels:
            if level in text_lower:
                experience_info.append(level)
                break

        return ", ".join(experience_info) if experience_info else "Not specified"

    def _extract_salary_info(self, text: str) -> str:
        """Извлекает информацию о зарплате"""
        salary_patterns = [
            r'\$(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*(?:-|to)\s*\$(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)',
            r'\$(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*(?:per\s*year|annually|p\.a\.)',
            r'(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*(?:USD|EUR|GBP)\s*(?:per\s*year|annually)',
            r'salary\s*(?:range)?\s*:\s*\$?(\d{1,3}(?:,\d{3})*)\s*(?:-|to)\s*\$?(\d{1,3}(?:,\d{3})*)',
            r'compensation\s*:\s*\$?(\d{1,3}(?:,\d{3})*)\s*(?:-|to)\s*\$?(\d{1,3}(?:,\d{3})*)'
        ]

        for pattern in salary_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                for match in matches:
                    if isinstance(match, tuple):
                        return f"${match[0]} - ${match[1]} per year"
                    else:
                        return f"${match} per year"

        return "Not specified / Competitive"

    def _extract_location_worktype(self, text_lower: str) -> tuple:
        """Извлекает информацию о локации и типе работы"""
        location = "Not specified"
        work_type = "Not specified"

        # Определение типа работы
        for w_type, patterns in self.work_type_patterns.items():
            for pattern in patterns:
                if pattern in text_lower:
                    work_type = w_type.capitalize()
                    break
            if work_type != "Not specified":
                break

        # Определение локации
        if "remote" in text_lower:
            location = "Remote"
        elif "hybrid" in text_lower:
            location = "Hybrid"
        elif "onsite" in text_lower or "office" in text_lower:
            location = "On-site"

        # Поиск конкретного города/страны
        location_keywords = [
            "london", "new york", "san francisco", "berlin", "amsterdam",
            "toronto", "sydney", "dublin", "paris", "singapore",
            "europe", "usa", "uk", "germany", "canada", "australia"
        ]

        for loc in location_keywords:
            if loc in text_lower:
                location = loc.title()
                break

        return location, work_type

    def _extract_key_skills(self, text_lower: str) -> List[str]:
        """Извлекает ключевые навыки"""
        skills = []

        # Ищем в формате "skills:" или "requirements:"
        lines = text_lower.split('\n')
        skills_section = False

        for line in lines:
            if any(keyword in line for keyword in ["skills:", "requirements:", "must have:", "should have:"]):
                skills_section = True
                continue

            if skills_section:
                if line.strip() and len(line.strip()) < 100:
                    skills.append(line.strip())
                elif line.strip() == '':
                    skills_section = False

        # Если не нашли структурированного раздела, извлекаем технологии
        if not skills:
            skills = self._extract_technologies(text_lower)

        return skills[:15]  # Ограничиваем количество

    def _generate_english_summary(self, title: str, requirements: List[str],
                                technologies: List[str], experience: str,
                                salary_info: str) -> str:
        """Генерирует краткое саммари на английском"""
        summary_parts = []

        summary_parts.append(f"Position: {title}")

        if experience and experience != "Not specified":
            summary_parts.append(f"Experience required: {experience}")

        if technologies:
            tech_list = ", ".join(technologies[:8])
            summary_parts.append(f"Key technologies: {tech_list}")

        if requirements:
            main_req = requirements[0] if len(requirements[0]) < 100 else requirements[0][:97] + "..."
            summary_parts.append(f"Main requirement: {main_req}")

        if salary_info and salary_info != "Not specified / Competitive":
            summary_parts.append(f"Salary: {salary_info}")

        return ". ".join(summary_parts)

    def _generate_russian_summary(self, english_summary: str) -> str:
        """Создает русскоязычное саммари"""
        # Простой перевод/адаптация ключевых фраз
        translation_map = {
            "Position:": "Должность:",
            "Experience required:": "Требуемый опыт:",
            "Key technologies:": "Ключевые технологии:",
            "Main requirement:": "Основное требование:",
            "Salary:": "Зарплата:",
            "Software Engineer": "Инженер-программист",
            "Developer": "Разработчик",
            "Senior": "Старший",
            "Junior": "Младший",
            "Remote": "Удалённая работа",
            "Hybrid": "Гибридный формат",
            "On-site": "Офис",
            "Full-time": "Полная занятость",
            "Part-time": "Частичная занятость",
            "Contract": "Контракт",
            "years": "лет",
            "year": "год",
            "experience": "опыт",
            "Python": "Python",
            "Java": "Java",
            "JavaScript": "JavaScript",
            "React": "React",
            "AWS": "AWS",
            "Docker": "Docker",
            "Kubernetes": "Kubernetes",
            "Not specified": "Не указано"
        }

        russian_summary = english_summary
        for eng, rus in translation_map.items():
            russian_summary = russian_summary.replace(eng, rus)

        return russian_summary

    def _calculate_confidence(self, requirements: List[str], technologies: List[str],
                            experience: str, skills: List[str]) -> float:
        """Рассчитывает уверенность в извлеченных данных"""
        confidence = 0.0

        # Требования (30 баллов)
        if len(requirements) >= 3:
            confidence += 30
        elif len(requirements) >= 1:
            confidence += 15

        # Технологии (30 баллов)
        if len(technologies) >= 5:
            confidence += 30
        elif len(technologies) >= 2:
            confidence += 15

        # Опыт (20 баллов)
        if experience and experience != "Not specified":
            confidence += 20

        # Навыки (20 баллов)
        if len(skills) >= 3:
            confidence += 20
        elif len(skills) >= 1:
            confidence += 10

        return min(confidence / 100, 1.0)

    def _create_csv_row(self, data: Dict, original_text: str, start_time: float) -> VacancySummary:
        """Создает строку для CSV"""
        processing_time = time.time() - start_time

        return VacancySummary(
            vacancy_id=data['vacancy_id'],
            vacancy_title=data['title'],
            extracted_requirements="; ".join(data['requirements'][:5]),
            extracted_technologies=", ".join(data['technologies'][:10]),
            extracted_experience=data['experience'],
            key_skills=", ".join(data['key_skills'][:10]),
            summary_english=data['english_summary'],
            summary_russian=data['russian_summary'],
            salary_info=data['salary_info'],
            location=data['location'],
            work_type=data['work_type'],
            confidence_score=data['confidence'],
            processing_date=datetime.now().strftime("%Y-%m-%d"),
            source_text_hash=hashlib.md5(original_text.encode()).hexdigest()[:10],
            source_text_length=len(original_text),
            processing_time=processing_time
        )

    def _create_error_row(self, vacancy_id: str, title: str, text: str,
                         error: str, start_time: float) -> VacancySummary:
        """Создает строку с ошибкой"""
        processing_time = time.time() - start_time

        return VacancySummary(
            vacancy_id=vacancy_id,
            vacancy_title=title,
            extracted_requirements=f"ERROR: {error[:100]}",
            extracted_technologies="",
            extracted_experience="",
            key_skills="",
            summary_english=f"Error processing vacancy: {error[:200]}",
            summary_russian=f"Ошибка обработки вакансии: {error[:200]}",
            salary_info="",
            location="",
            work_type="",
            confidence_score=0.0,
            processing_date=datetime.now().strftime("%Y-%m-%d"),
            source_text_hash=hashlib.md5(text.encode()).hexdigest()[:10],
            source_text_length=len(text),
            processing_time=processing_time
        )

    def _save_to_csv(self, df: pd.DataFrame, filename: str):
        """Сохраняет DataFrame в CSV файл"""
        # Создаем понятные имена колонок на русском
        russian_columns = {
            'vacancy_id': 'ID_вакансии',
            'vacancy_title': 'Название_вакансии',
            'extracted_requirements': 'Извлеченные_требования',
            'extracted_technologies': 'Технологии',
            'extracted_experience': 'Требуемый_опыт',
            'key_skills': 'Ключевые_навыки',
            'summary_english': 'Саммари_английский',
            'summary_russian': 'Саммари_русский',
            'salary_info': 'Информация_о_зарплате',
            'location': 'Локация',
            'work_type': 'Тип_работы',
            'confidence_score': 'Уверенность_извлечения',
            'processing_date': 'Дата_обработки',
            'source_text_hash': 'Хэш_исходного_текста',
            'source_text_length': 'Длина_исходного_текста',
            'processing_time': 'Время_обработки_сек'
        }

        df_renamed = df.rename(columns=russian_columns)
        df_renamed.to_csv(filename, index=False, encoding='utf-8-sig')

        if self.verbose:
            print(f"\n[СОХРАНЕНИЕ] CSV файл создан: {filename}")
            print(f"[РАЗМЕР] {len(df)} строк, {len(df.columns)} колонок")

    def _print_statistics(self, processed: int, errors: int, df: pd.DataFrame):
        """Выводит статистику обработки"""
        if self.verbose:
            print("СТАТИСТИКА ОБРАБОТКИ")
            total = processed + errors

            print(f"Всего вакансий: {total}")
            print(f"Успешно обработано: {processed} ({processed/total*100:.1f}%)")
            print(f"С ошибками: {errors} ({errors/total*100:.1f}%)")

            if processed > 0 and len(df) > 0:
                successful_df = df[df['confidence_score'] > 0]
                if len(successful_df) > 0:
                    avg_confidence = successful_df['confidence_score'].mean()
                    avg_processing = successful_df['processing_time'].mean()
                    avg_length = successful_df['source_text_length'].mean()

                    print(f"\nСредняя уверенность: {avg_confidence:.2%}")
                    print(f"Среднее время обработки: {avg_processing:.2f} сек")
                    print(f"Средняя длина текста: {avg_length:.0f} символов")

                    # Топ-5 технологий
                    all_tech = []
                    for tech_str in successful_df['extracted_technologies']:
                        if pd.notna(tech_str):
                            all_tech.extend([t.strip() for t in tech_str.split(',')])

                    if all_tech:
                        from collections import Counter
                        tech_counter = Counter(all_tech)
                        print("\nТоп-5 технологий в вакансиях:")
                        for tech, count in tech_counter.most_common(5):
                            print(f"  {tech}: {count} упоминаний")

# Функция для быстрого создания суммаризатора
def create_summarizer(verbose: bool = True) -> VacancyToCSVSummarizer:
    """Создает экземпляр суммаризатора"""
    return VacancyToCSVSummarizer(verbose=verbose)